In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/10/01 06:00:05 WARN Utils: Your hostname, nabill-IdeaPad-3-14IML05 resolves to a loopback address: 127.0.1.1; using 192.168.11.111 instead (on interface wlp0s20f3)
26/10/01 06:00:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 06:00:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as _sum, count

spark = SparkSession.builder.appName("Tugas6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("Jumlah baris df_trx:", df_trx.count())
df_trx.printSchema()

print("Jumlah baris df_produk:", df_produk.count())
df_produk.printSchema()

print("Jumlah baris df_ulasan:", df_ulasan.count())
df_ulasan.printSchema()

Jumlah baris df_trx: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah baris df_produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah baris df_ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [6]:
df_trx_ulasan = df_trx.join(df_ulasan, on="order_id", how="left")

df_gabungan = df_trx_ulasan.join(df_produk, on="product_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_gabungan.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

In [7]:
df_gabungan = df_gabungan.withColumn(
    "ada_ulasan", when(col("rating").isNotNull(), True).otherwise(False)
)

df_gabungan = df_gabungan.na.fill({"rating": 0})

df_gabungan.select("order_id", "rating", "ada_ulasan").show(10)

+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|     0|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|     0|     false|
|     TX5|     5|      true|
|     TX6|     4|      true|
|     TX7|     0|     false|
|     TX8|     5|      true|
|     TX9|     0|     false|
+--------+------+----------+
only showing top 10 rows



In [8]:
df_gabungan.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/nabill/tugas6/hasil_etl"
)

In [10]:
!hdfs dfs -ls -R /user/nabill/tugas6/hasil_etl

-rw-r--r--   3 nabill supergroup          0 2026-10-01 06:04 /user/nabill/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - nabill supergroup          0 2026-10-01 06:03 /user/nabill/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 nabill supergroup      16366 2026-10-01 06:04 /user/nabill/tugas6/hasil_etl/kategori=Elektronik/part-00000-189dca76-ffcb-49f8-b35b-f19eec2e6882.c000.snappy.parquet
drwxr-xr-x   - nabill supergroup          0 2026-10-01 06:04 /user/nabill/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 nabill supergroup      11133 2026-10-01 06:04 /user/nabill/tugas6/hasil_etl/kategori=Fashion/part-00000-189dca76-ffcb-49f8-b35b-f19eec2e6882.c000.snappy.parquet
drwxr-xr-x   - nabill supergroup          0 2026-10-01 06:04 /user/nabill/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 nabill supergroup      10392 2026-10-01 06:04 /user/nabill/tugas6/hasil_etl/kategori=Kesehatan/part-00000-189dca76-ffcb-49f8-b35b-f19eec2e6882.c000.snappy.parquet
drwxr-xr-x   - nabill supergroup      

In [11]:
df_verifikasi = spark.read.parquet("hdfs://localhost:9000/user/nabill/tugas6/hasil_etl")
print("Jumlah baris hasil baca kembali:", df_verifikasi.count())

Jumlah baris hasil baca kembali: 5000


In [12]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("tugas6").getOrCreate()

try:
    df_etl = spark.read.parquet("hdfs://localhost:9000/user/nabill/tugas6/hasil_etl")
except Exception:
    df_etl = spark.createDataFrame(df_final)

# gunakan DataFrame API
insight = (
    df_etl
    .groupBy("kategori")
    .agg(
        F.count("*").alias("total_transaksi"),
        F.sum(F.col("ada_ulasan").cast("int")).alias("transaksi_berulasan")
    )
    .withColumn(
        "persen_ulasan",
        F.round(F.col("transaksi_berulasan") / F.col("total_transaksi") * 100, 2)
    )
    .orderBy("persen_ulasan")
)

insight.show(truncate=False)

print("Kategori dengan persentase transaksi berulasan paling rendah:")
insight.limit(1).show(truncate=False)

+------------+---------------+-------------------+-------------+
|kategori    |total_transaksi|transaksi_berulasan|persen_ulasan|
+------------+---------------+-------------------+-------------+
|Makanan     |536            |369                |68.84        |
|Kesehatan   |961            |662                |68.89        |
|Fashion     |1035           |726                |70.14        |
|Rumah Tangga|815            |575                |70.55        |
|Elektronik  |1653           |1168               |70.66        |
+------------+---------------+-------------------+-------------+

Kategori dengan persentase transaksi berulasan paling rendah:
+--------+---------------+-------------------+-------------+
|kategori|total_transaksi|transaksi_berulasan|persen_ulasan|
+--------+---------------+-------------------+-------------+
|Makanan |536            |369                |68.84        |
+--------+---------------+-------------------+-------------+

